<div dir="rtl">

# מדידת ההתרעות על UCF-Crime, ב-Colab

המחברת מריצה את כל שלבי המדידה על GPU:

1. מורידה מתוך הארכיון של UCF-Crime **רק** את הסרטונים שיש להם תיוג ב-`eval/ground_truth_uca.json`: 107 סרטונים של שוד, הצתה, ירי וסרטונים רגילים, עם התרחישים knife / weapon / fire / snatch / package.
2. מריצה עליהם את מה שהמצלמה החיה מריצה, כלומר שלד וגלאים (`live/perceive.py`).
3. משווה בין השיטות לכל תרחיש (`live/eval_alerts.py`): למשל "רואים סכין" מול "מרים סכין", ו"רואים אש" מול "מישהו מצית". המדדים הם recall, התרעות שווא לשעה, וזמן עד התרעה.

**לא צריך לתייג כלום.** התיוג של הסרטונים האלה כבר קיים בריפו.

**לפני שמתחילים:** Runtime → Change runtime type → **GPU**. ואז Runtime → Run all.

**זמן:** השרת של UCF איטי, בערך 0.7MB/s לחיבור. תא 3 מדפיס כמה GB צריך להוריד. ההורדה וה-perception נשמרים ב-Drive ונמשכים מאיפה שנעצרו, כך שאם Colab מתנתק פשוט מריצים שוב.

</div>

In [ ]:
# 1. GPU + Drive (videos, perception cache and results are kept in MyDrive/live_eval)
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/live_eval'
os.makedirs(DRIVE, exist_ok=True)

In [ ]:
# 2. Code + dependencies (Colab already has torch with CUDA)
BRANCH = 'live-action-alerts'
!rm -rf /content/final_project
!git clone -q -b {BRANCH} https://github.com/Talyaavraham/final_project.git /content/final_project
%cd /content/final_project/backend
!pip install -q ultralytics lap remotezip truststore
!ls security.pt live/perceive.py live/eval_alerts.py eval/ground_truth_uca.json

In [ ]:
# 3. Which videos: the names in the ground truth, found in the UCF-Crime archive (only its index is read)
import json, truststore
truststore.inject_into_ssl()            # the UCF server sends an incomplete certificate chain
from remotezip import RemoteZip
URL = 'https://www.crcv.ucf.edu/data1/chenchen/UCF_Crimes.zip'
gt = json.load(open('eval/ground_truth_uca.json'))['videos']
want = {os.path.splitext(v)[0] for v in gt}
with RemoteZip(URL, timeout=(30, 120)) as z:
    infos = {os.path.splitext(os.path.basename(i.filename))[0]: i for i in z.infolist() if i.filename.endswith('.mp4')}
sel = {n: {'path': infos[n].filename, 'bytes': infos[n].file_size} for n in sorted(want) if n in infos}
missing = sorted(want - set(sel))
os.makedirs('eval/data', exist_ok=True)
json.dump(sel, open('eval/data/ucf_selection.json', 'w'), indent=1)
print(f"{len(sel)} of {len(want)} videos are in the archive, {sum(v['bytes'] for v in sel.values()) / 1e9:.1f} GB")
if missing:
    print('not found:', missing[:10])

In [ ]:
# 4. Download them to Drive (resumable: run again after a disconnect)
os.makedirs(f'{DRIVE}/ucf_videos', exist_ok=True)
if not os.path.islink('eval/data/ucf'):
    !rm -rf eval/data/ucf
    os.symlink(f'{DRIVE}/ucf_videos', 'eval/data/ucf')
!python eval/fetch_ucf.py --workers 4
!ls eval/data/ucf | wc -l

In [ ]:
# 5. Perception: the live pose + detectors on every frame at FPS (slow part, cached on Drive, resumable).
#    FPS = the rate the live system really runs at: laptop CPU about 1-3, GPU 10+. Run again with another FPS to compare.
FPS = 3
os.environ['LIVE_DEVICE'] = '0'
PERC = f'{DRIVE}/perception_fps{FPS}'
!python live/perceive.py --gt eval/ground_truth_uca.json --videos eval/data/ucf --out {PERC} --fps {FPS}

In [ ]:
# 6. Evaluate every method of live/scenarios.json (seconds). Results -> live/results and Drive.
!python live/eval_alerts.py --gt eval/ground_truth_uca.json --perception {PERC} --name ucf_fps{FPS}
!cp live/results/ucf_fps{FPS}.* {DRIVE}/

In [ ]:
# 7. The same with other detector thresholds (more alerts vs fewer false alarms)
for c in (0.2, 0.3, 0.4, 0.5):
    print(f'\n===== conf {c}')
    !python live/eval_alerts.py --gt eval/ground_truth_uca.json --perception {PERC} --conf {c} --name ucf_fps{FPS}_conf{c} | head -25
!cp live/results/ucf_fps{FPS}_conf* {DRIVE}/

<div dir="rtl">

**סרטונים שלכן (לא חובה):** אם תייגתן סרטונים במסך התיוג, מעלים ל-`MyDrive/live_eval/own/` את הקובץ `labels.json` (מ-`backend/live/data/`) ואת הסרטונים עצמם, ומריצים את התא הבא.

</div>

In [ ]:
# 8. (optional) your own labelled videos from MyDrive/live_eval/own/
OWN = f'{DRIVE}/own'
if os.path.isfile(f'{OWN}/labels.json'):
    !python live/perceive.py --videos {OWN} --out {DRIVE}/perception_own_fps{FPS} --fps {FPS}
    !python live/eval_alerts.py --gt {OWN}/labels.json --perception {DRIVE}/perception_own_fps{FPS} --name own_fps{FPS}
    !cp live/results/own_fps{FPS}.* {DRIVE}/
else:
    print('no', OWN + '/labels.json')

In [ ]:
# 9. Download the result files to this computer
from google.colab import files
import glob
for p in sorted(glob.glob('live/results/*')):
    files.download(p)